# Build Toy MLP for multilabel prediciton using ft embeddings

In [43]:
"""Load Data
Structure:
    1. Imports, Variables, Functions
    2. Load Data
"""

# 1. Imports, Variables, Functions
# imports
import pandas as pd, numpy as np, os, sys
import anndata as ad
import logging
from typing import *
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.metrics import confusion_matrix, classification_report
import sys
sys.path.append(os.path.join("..", ".."))
from src.utils import utils as ut
from src.utils import viz as vz
logging.basicConfig(level=logging.INFO)


import torch

torch.cuda.empty_cache()   # releases cached memory back to the GPU
torch.cuda.synchronize()   # optional, waits for all streams to finish


In [2]:

# variables
run_dir = os.path.join("..","..","outputs","run-25-08-08-03") # doid - grouped multilabel CLS_MULTILABEL
output_dir = os.path.join(run_dir, "outputs")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
split_idx = 0
# functions


def load_run_output(input_dir: str) -> tuple:
    """Load the output of a run
    Args:
        input_dir (str): path to the run output directory
    Returns:
        loaded_variables (tuple): tuple of loaded variables
    """

    variables_to_load = [
        # "split",
        "predictions_test",
        "labels_test",
        "results_test",
        "all_outputs_test",
        "predictions_valid",
        "labels_valid",
        "results_valid",
        "all_outputs_valid",
        "predictions_train",
        "labels_train",
        "results_train",
        "all_outputs_train",
        "adata_orig",
        "id2type",
        "train_indices",
        "valid_indices",
    ]

    # initialize loaded variables as an empty tuple
    loaded_variables = ()

    # loop through variables
    for variable in variables_to_load:
        if variable.startswith("adata_"):
            if False:
                # load everything
                loaded_variable = ad.read_h5ad(
                    os.path.join(input_dir, f"{variable}.h5ad")
                )

            else:
                # do not load everything
                loaded_variable = ad.read_h5ad(
                    os.path.join(input_dir, f"{variable}.h5ad"), backed="r"
                )

        else:
            with open(os.path.join(input_dir, f"{variable}.pkl"), "rb") as f:
                loaded_variable = pickle.load(f)

        # add the loaded variable to the tuple
        loaded_variables += (loaded_variable,)

    print(f"Nº of loaded variables {len(loaded_variables)}")

    return loaded_variables


# 2. Load Data
(
    # split,
    predictions_test,
    labels_test,
    results_test,
    all_outputs_test,
    predictions_valid,
    labels_valid,
    results_valid,
    all_outputs_valid,
    predictions_train,
    labels_train,
    results_train,
    all_outputs_train,
    adata_orig,
    id2type,
    train_indices,
    valid_indices,
) = load_run_output(run_dir)

# load json

with open(os.path.join(run_dir, "parameters.json"), "r") as f:
    parameters = json.load(f)

for k, v in parameters.items():
    print(f"{k}: {v}")

# 2. Load Data
# load all adata
adata_test =  ad.read_h5ad(
    os.path.join(run_dir, f"adata_test_{split_idx+1}.h5ad"), backed="r"
)
adata_valid =  ad.read_h5ad(
    os.path.join(run_dir, f"adata_valid_{split_idx+1}.h5ad"), backed="r"
)
adata_train =  ad.read_h5ad(
    os.path.join(run_dir, f"adata_train_{split_idx+1}.h5ad"), backed="r"
)

# load scGPT embeddings
embeddings_test = vz.merge_embeddings(all_outputs_test[split_idx])
embeddings_valid = vz.merge_embeddings(all_outputs_valid[split_idx])
embeddings_train = vz.merge_embeddings(all_outputs_train[split_idx])



Nº of loaded variables 16
data_path: /aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-05-07-01/data.h5ad
max_seq_len: 3501
batch_size: 16
gene_presence_pct: 0.9
benchmark_data: False
split_type: stratified
val_split_type: random
n_splits: 3
n_tested_splits: 3
epochs: 50
gene_filtering: top_presence
sample_presence_pct: 0.3
MLM: False
CLS: False
CLS_multilabel: True
DAB: False
ADV: False
CCE: False
ecs_thres: 0.0
dab_weight: 0.0
use_fast_transformer: True
output_attentions: False
INPUT_BATCH_LABELS: False
do_combat: False
ontology: do


In [3]:
sys.path.append("../../")
from src.utils import utils as u

# get Disease Ontology graph
do_g = u.load_do_graph()

# get sanchez IC
doid_2_ic = u.get_sanchez_ic(do_g)

# get nodes
_class_nodes = u.get_n_lowest_ic_nodes(doid_2_ic, 50)

# Generate multilabel vectors for level 1 nodes
Y_multilabel, _class_nodes = u.generate_multilabel_vectors(adata_train, do_g, _class_nodes)
print(f"Generated multilabel vectors for level 1 nodes with shape {Y_multilabel.shape}")

# Clean multilabel vectors by removing nodes with no samples
Y_multilabel, _class_nodes = u.clean_multilabel_vectors(Y_multilabel, _class_nodes)
print(f"Cleaned multilabel vectors for top 50 nodes with shape {Y_multilabel.shape}")

# Check the multilabel vector for level 1 nodes
u.check_multilabel_vector(Y_multilabel, _class_nodes, do_g)

Number of DO leaves: 9018
Generated multilabel vectors for level 1 nodes with shape (11921, 50)
Cleaned multilabel vectors for top 50 nodes with shape (11921, 27)
215 samples	Node DOID:0014667 - disease of metabolism
804 samples	Node DOID:0050117 - disease by infectious agent
191 samples	Node DOID:0050155 - sensory system disease
74 samples	Node DOID:0050177 - monogenic disease
958 samples	Node DOID:0050686 - organ system cancer
716 samples	Node DOID:0050687 - cell type cancer
74 samples	Node DOID:0050735 - X-linked monogenic disease
356 samples	Node DOID:0080000 - muscular disease
480 samples	Node DOID:1287 - cardiovascular system disease
1674 samples	Node DOID:14566 - disease of cellular proliferation
229 samples	Node DOID:15 - reproductive system disease
205 samples	Node DOID:150 - disease of mental health
383 samples	Node DOID:1579 - respiratory system disease
948 samples	Node DOID:16 - integumentary system disease
1674 samples	Node DOID:162 - cancer
3859 samples	Node DOID:17 - mus

In [4]:
# Generate multilabel vectors for level 1 nodes
Y_multilabel_valid, _class_nodes_valid = u.generate_multilabel_vectors(adata_valid, do_g, _class_nodes)
print(f"Generated multilabel vectors for level 1 nodes with shape {Y_multilabel_valid.shape}")

Generated multilabel vectors for level 1 nodes with shape (1325, 27)


In [ ]:
# Generate multilabel vectors for level 1 nodes
Y_multilabel_test, _class_nodes_test = u.generate_multilabel_vectors(adata_test, do_g, _class_nodes)
print(f"Generated multilabel vectors for level 1 nodes with shape {Y_multilabel_test.shape}")

Generated multilabel vectors for level 1 nodes with shape (7306, 27)


In [49]:
# Build small torch MLP for multilabel prediction using ft embeddings
# 3. Build MLP
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

# model
class MLP_multilabel(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)           # no sigmoid here
        return logits


class MLP_multilabel(nn.Module):
    def __init__(self, layer_sizes:List[int]):
        super().__init__()
        self.layers = nn.ModuleList(nn.Linear(layer_sizes[i], layer_sizes[i + 1]) for i in range(len(layer_sizes) - 1))
    def forward(self, x):
        for layer in self.layers[:-1]:
            x = F.relu(layer(x))
        logits = self.layers[-1](x)
        return logits
# metrics

def multilabel_metrics_from_logits(logits, y_true, threshold=0.5):
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).int()
        y_true = y_true.int()

        # Labelwise accuracy (mean over samples, then labels)
        labelwise_acc = (preds == y_true).float().mean().item()

        # Micro-F1
        tp = (preds & y_true).sum().item()
        fp = (preds & (1 - y_true)).sum().item()
        fn = ((1 - preds) & y_true).sum().item()
        precision = tp / (tp + fp + 1e-12)
        recall    = tp / (tp + fn + 1e-12)
        f1_micro  = 2 * precision * recall / (precision + recall + 1e-12)


        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= threshold)
        y_true = y_true.cpu().numpy()

        ap = average_precision_score(y_true, probs, average="micro")
        auroc = roc_auc_score(y_true, probs, average="micro")

        return {"labelwise_acc": labelwise_acc, "f1_micro": f1_micro, "auroc": auroc, "auprc": ap}




def train_epoch(model, train_loader, criterion, optimizer)->Tuple[float]:
    model.train()  # set model to training mode
    train_loss = 0.0
    train_n = 0
    for xb, yb in train_loader:
        # Move to device
        xb = xb.to(device)
        yb = yb.to(device)

        # define optimizer step
        optimizer.zero_grad(set_to_none=True)

        # forward pass 
        logits = model(xb)
        loss = criterion(logits, yb)

        # backward pass
        loss.backward() # computes gradients
        optimizer.step()    # update model weights

        # save loss 
        bs = xb.size(0)
        train_n += bs
        train_loss += loss.item() * bs # multiply by batch size - loss is average!

    return train_loss / max(train_n, 1)  # return average loss across samples

def compute_metrics(model, loader, criterion, threshold=0.5)->Tuple[float, Dict]:
    model.eval() # set model to evaluation mode
    val_n = 0
    val_loss = 0.0
    metrics = {"labelwise_acc": 0.0, "f1_micro": 0.0, "auroc": 0.0, "auprc": 0.0}
    with torch.no_grad(): # no gradient computation
        for xb, yb in loader:
            # Move to device
            xb = xb.to(device)
            yb = yb.to(device)
            
            # forward pass 
            logits = model(xb)
            loss = criterion(logits, yb)

            # save loss
            bs = xb.size(0)
            val_n += bs
            val_loss += loss.item() * bs

            # compute metrics
            m = multilabel_metrics_from_logits(logits, yb, threshold=threshold)
            for k in metrics:
                metrics[k] += m[k] * bs

    # average metrics
    for k in metrics:
        metrics[k] /= val_n
    return val_loss / val_n, metrics
        


# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# define MLP model
mlp_mod = MLP_multilabel(
    [embeddings_train.shape[1],embeddings_train.shape[1]//2, Y_multilabel.shape[1]]
).to(device)

# define loss function
criterion = nn.BCEWithLogitsLoss()

# define optimizer
optimizer = torch.optim.Adam(mlp_mod.parameters(), lr=1e-3)

# StepLR: decay LR by gamma every step_size epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

# Convert data to PyTorch tensors
train_data = TensorDataset(
    torch.tensor(embeddings_train, dtype=torch.float32),
    torch.tensor(Y_multilabel, dtype=torch.float32),
)
valid_data = TensorDataset(
    torch.tensor(embeddings_valid, dtype=torch.float32),
    torch.tensor(Y_multilabel_valid, dtype=torch.float32),
)
test_data = TensorDataset(
    torch.tensor(embeddings_test, dtype=torch.float32),
    torch.tensor(Y_multilabel_test, dtype=torch.float32),
)

# DataLoaders
train_loader = DataLoader(train_data, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_data, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)


# Train & Validate
epochs = 100
threshold = 0.5
best_val_loss = float("inf")



patience = 0
for epoch in range(1, epochs + 1):
   
    # Train
    avg_train_loss = train_epoch(model=mlp_mod, train_loader=train_loader, criterion=criterion, optimizer=optimizer)

    # Validation
    avg_val_loss, metrics = compute_metrics(model=mlp_mod, loader=valid_loader, criterion=criterion, threshold=threshold)

    avg_f1_micro = metrics["f1_micro"]
    avg_labelwise_acc = metrics["labelwise_acc"]

    # report performance
    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"val_loss={avg_val_loss:.4f} | "
        f"lr={scheduler.get_last_lr()[0]:.6f}", end=" | "
    )
    for k, v in metrics.items():
        print(f"val_{k}: {v:.4f}", end=" | ")
    print()  # new line


    # Step the LR based on average validation loss
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        patience = 0
        best_val_loss = avg_val_loss
    else:
        patience += 1
    if patience >= 10:
        break


Epoch 01 | train_loss=0.1120 | val_loss=0.0635 | lr=0.001000 | val_labelwise_acc: 0.9805 | val_f1_micro: 0.8909 | val_auroc: 0.9874 | val_auprc: 0.9409 | 
Epoch 02 | train_loss=0.0446 | val_loss=0.0614 | lr=0.001000 | val_labelwise_acc: 0.9814 | val_f1_micro: 0.8956 | val_auroc: 0.9885 | val_auprc: 0.9423 | 
Epoch 03 | train_loss=0.0420 | val_loss=0.0613 | lr=0.001000 | val_labelwise_acc: 0.9815 | val_f1_micro: 0.8960 | val_auroc: 0.9885 | val_auprc: 0.9431 | 
Epoch 04 | train_loss=0.0411 | val_loss=0.0619 | lr=0.001000 | val_labelwise_acc: 0.9806 | val_f1_micro: 0.8923 | val_auroc: 0.9889 | val_auprc: 0.9442 | 
Epoch 05 | train_loss=0.0405 | val_loss=0.0626 | lr=0.001000 | val_labelwise_acc: 0.9813 | val_f1_micro: 0.8955 | val_auroc: 0.9884 | val_auprc: 0.9428 | 
Epoch 06 | train_loss=0.0393 | val_loss=0.0630 | lr=0.001000 | val_labelwise_acc: 0.9816 | val_f1_micro: 0.8972 | val_auroc: 0.9887 | val_auprc: 0.9447 | 
Epoch 07 | train_loss=0.0386 | val_loss=0.0634 | lr=0.001000 | val_lab

In [50]:

# train
_, metrics = compute_metrics(model=mlp_mod, loader=train_loader, criterion=criterion, threshold=threshold)
for k, v in metrics.items():
    print(f"train_{k}: {v:.4f}", end=" | ")
print() 

# val
_, metrics = compute_metrics(model=mlp_mod, loader=valid_loader, criterion=criterion, threshold=threshold)
for k, v in metrics.items():
    print(f"val_{k}: {v:.4f}", end=" | ")
print() 
# test
_, metrics = compute_metrics(model=mlp_mod, loader=test_loader, criterion=criterion, threshold=threshold)
for k, v in metrics.items():
    print(f"test_{k}: {v:.4f}", end=" | ")
print() 

train_labelwise_acc: 0.9879 | train_f1_micro: 0.9335 | train_auroc: 0.9969 | train_auprc: 0.9797 | 
val_labelwise_acc: 0.9811 | val_f1_micro: 0.8954 | val_auroc: 0.9881 | val_auprc: 0.9428 | 
test_labelwise_acc: 0.9418 | test_f1_micro: 0.6353 | test_auroc: 0.8674 | test_auprc: 0.6254 | 


In [115]:
def compute_metrics_mc(model, loader, criterion, threshold=0.5) -> Tuple[float, Dict]:
    """Compute Metrics for Multiclass Classification (softmax + argmax)."""
    model.eval()
    val_n, val_loss = 0, 0.0
    metrics = {"labelwise_acc": 0.0, "f1_micro": 0.0, "auroc": 0.0, "auprc": 0.0}
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            bs = xb.size(0)
            val_n += bs
            val_loss += loss.item() * bs

            m = metrics_from_logits_mc(logits, yb)  # threshold unused for MC
            for k in metrics:
                metrics[k] += m[k] * bs

    for k in metrics:
        metrics[k] /= max(val_n, 1)
    return val_loss / max(val_n, 1), metrics




# 2) Multiclass metrics: softmax + argmax; OVR AUROC/AUPRC
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score
import torch.nn.functional as F

def metrics_from_logits_mc(logits, y_true):
    """
    Multiclass metrics:
      - probs via softmax
      - predictions via argmax
      - AUROC/AUPRC computed one-vs-rest on softmax probs (micro-averaged)
    Keeps keys: labelwise_acc (overall accuracy), f1_micro, auroc, auprc.
    """
    with torch.no_grad():
        probs = F.softmax(logits, dim=1)                # [B, K]
        if y_true.dim() == 1:
            y_idx = y_true.long()                       # [B]
            K = probs.size(1)
            y_onehot = F.one_hot(y_idx, num_classes=K)  # [B, K]

        preds_idx = probs.argmax(dim=1)

        # Overall accuracy (reuse existing key name)
        acc = (preds_idx == y_idx).float().mean().item()

        # Micro-F1 over class indices
        f1_micro = f1_score(y_idx.cpu().numpy(), preds_idx.cpu().numpy(), average="micro")

        # OVR AU(PR)C (micro). Guard against degenerate batches.
        probs_np = probs.detach().cpu().numpy()
        y_oh_np  = y_onehot.detach().cpu().numpy()

        try:
            auprc = average_precision_score(y_oh_np, probs_np, average="micro")
        except ValueError:
            auprc = float("nan")

        try:
            auroc = roc_auc_score(y_oh_np, probs_np, average="micro", multi_class="ovr")
        except ValueError:
            auroc = float("nan")

        return {"labelwise_acc": acc, "f1_micro": f1_micro, "auroc": auroc, "auprc": auprc}



_disease_train = adata_train.obs["do_id_study"].to_list()
_disease_valid = adata_valid.obs["do_id_study"].to_list()
_disease_test = adata_test.obs["do_id_study"].to_list()

_unique_labels = sorted(set(_disease_train + _disease_valid + _disease_test))
_label_2_id = {l: i for i, l in enumerate(_unique_labels)}

Y_train = torch.tensor([_label_2_id[c] for c in _disease_train], dtype=torch.long)
Y_valid = torch.tensor([_label_2_id[c] for c in _disease_valid], dtype=torch.long)
Y_test = torch.tensor([_label_2_id[c] for c in _disease_test], dtype=torch.long)
n_labels = len(_unique_labels)

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# define MLP model
mlp_mod = MLP_multilabel(
    [embeddings_train.shape[1],
    embeddings_train.shape[1]//2, n_labels
    ]
).to(device)

# define loss function
criterion = nn.CrossEntropyLoss()   # it takes y as long tensor, not multilabel ! 
                                    # y true therefore is the index of true value !

# define optimizer
optimizer = torch.optim.Adam(mlp_mod.parameters(), lr=1e-3)

# StepLR: decay LR by gamma every step_size epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

# Convert data to PyTorch tensors
train_data = TensorDataset(
    torch.tensor(embeddings_train, dtype=torch.float32),
    torch.tensor(Y_train, dtype=torch.long),
)
valid_data = TensorDataset(
    torch.tensor(embeddings_valid, dtype=torch.float32),
    torch.tensor(Y_valid, dtype=torch.long),
)
test_data = TensorDataset(
    torch.tensor(embeddings_test, dtype=torch.float32),
    torch.tensor(Y_test, dtype=torch.long),
)

# DataLoaders
train_loader = DataLoader(train_data, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_data, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)


# Train & Validate
epochs = 100
threshold = 0.5
best_val_loss = float("inf")



patience = 0
for epoch in range(1, epochs + 1):
   
    # Train
    avg_train_loss = train_epoch(model=mlp_mod, train_loader=train_loader, criterion=criterion, optimizer=optimizer)

    # Validation
    avg_val_loss, metrics = compute_metrics_mc(model=mlp_mod, loader=valid_loader, criterion=criterion, threshold=threshold)

    avg_f1_micro = metrics["f1_micro"]
    avg_labelwise_acc = metrics["labelwise_acc"]

    # report performance
    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"val_loss={avg_val_loss:.4f} | "
        f"lr={scheduler.get_last_lr()[0]:.6f}", end=" | "
    )
    for k, v in metrics.items():
        print(f"val_{k}: {v:.4f}", end=" | ")
    print()  # new line


    # Step the LR based on average validation loss
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        patience = 0
        best_val_loss = avg_val_loss
    else:
        patience += 1
    if patience >= 10:
        break

/tmp/ipykernel_2323893/1591211638.py:107: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(Y_train, dtype=torch.long),
/tmp/ipykernel_2323893/1591211638.py:111: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(Y_valid, dtype=torch.long),
/tmp/ipykernel_2323893/1591211638.py:115: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(Y_test, dtype=torch.long),


Epoch 01 | train_loss=2.0198 | val_loss=1.4634 | lr=0.001000 | val_labelwise_acc: 0.6272 | val_f1_micro: 0.6272 | val_auroc: 0.9894 | val_auprc: 0.7136 | 
Epoch 02 | train_loss=1.0633 | val_loss=1.2152 | lr=0.001000 | val_labelwise_acc: 0.6989 | val_f1_micro: 0.6989 | val_auroc: 0.9925 | val_auprc: 0.7794 | 
Epoch 03 | train_loss=0.8827 | val_loss=1.1405 | lr=0.001000 | val_labelwise_acc: 0.7283 | val_f1_micro: 0.7283 | val_auroc: 0.9931 | val_auprc: 0.8001 | 
Epoch 04 | train_loss=0.7947 | val_loss=1.1022 | lr=0.001000 | val_labelwise_acc: 0.7442 | val_f1_micro: 0.7442 | val_auroc: 0.9934 | val_auprc: 0.8136 | 
Epoch 05 | train_loss=0.7488 | val_loss=1.0996 | lr=0.001000 | val_labelwise_acc: 0.7419 | val_f1_micro: 0.7419 | val_auroc: 0.9934 | val_auprc: 0.8171 | 
Epoch 06 | train_loss=0.7092 | val_loss=1.0730 | lr=0.001000 | val_labelwise_acc: 0.7562 | val_f1_micro: 0.7562 | val_auroc: 0.9935 | val_auprc: 0.8238 | 
Epoch 07 | train_loss=0.6809 | val_loss=1.0959 | lr=0.001000 | val_lab

In [ ]:
_disease_train = adata_train.obs["do_id_study"].to_list()
_disease_valid = adata_valid.obs["do_id_study"].to_list()
_disease_test = adata_test.obs["do_id_study"].to_list()

_unique_labels = sorted(set(_disease_train + _disease_valid + _disease_test))
_lable_2_id = {l: i for i, l in enumerate(_unique_labels)}

Y_train = torch.tensor([_lable_2_id[c] for c in _disease_train], dtype=torch.long)


11921

In [84]:
torch.tensor([_lable_2_id[c] for c in _disease_train], dtype=torch.long)

tensor([119,  10,  85,  ..., 153, 113,  66])

In [75]:
len(_unique_labels)

210

In [74]:
Y_train.shape

torch.Size([11921])

In [71]:
Y_multilabel.shape[1]

27

In [90]:
# train
_, metrics = compute_metrics_mc(model=mlp_mod, loader=train_loader, criterion=criterion, threshold=threshold)
for k, v in metrics.items():
    print(f"train_{k}: {v:.4f}", end=" | ")
print() 

# val
_, metrics = compute_metrics_mc(model=mlp_mod, loader=valid_loader, criterion=criterion, threshold=threshold)
for k, v in metrics.items():
    print(f"val_{k}: {v:.4f}", end=" | ")
print() 
# test
_, metrics = compute_metrics_mc(model=mlp_mod, loader=test_loader, criterion=criterion, threshold=threshold)
for k, v in metrics.items():
    print(f"test_{k}: {v:.4f}", end=" | ")
print() 

train_labelwise_acc: 0.8750 | train_f1_micro: 0.8750 | train_auroc: 0.9993 | train_auprc: 0.9430 | 
val_labelwise_acc: 0.7570 | val_f1_micro: 0.7570 | val_auroc: 0.9922 | val_auprc: 0.8214 | 
test_labelwise_acc: 0.2836 | test_f1_micro: 0.2836 | test_auroc: 0.8170 | test_auprc: 0.2410 | 


In [70]:
Y_train.shape

torch.Size([11921])

In [ ]:
import numpy as np

min([np.nan + 1])

nan

In [111]:
mlp_mod.eval()
val_n, val_loss = 0, 0.0
metrics = {"labelwise_acc": 0.0, "f1_micro": 0.0, "auroc": 0.0, "auprc": 0.0}
with torch.no_grad():
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = mlp_mod(xb)
        loss = criterion(logits, yb)

        bs = xb.size(0)
        val_n += bs
        val_loss += loss.item() * bs

        m = metrics_from_logits_mc(logits, yb)  # threshold unused for MC
        for k in metrics:
            metrics[k] += m[k] * bs

for k in metrics:
    metrics[k] /= max(val_n, 1)

In [ ]:
probs = F.softmax(logits, dim=1)                # [B, K]
probs.max(dim=0)

torch.return_types.max(
values=tensor([9.9193e-01, 5.8339e-02, 3.9261e-07, 4.6396e-03, 9.2188e-03, 8.3861e-07,
        1.6336e-05, 2.1649e-05, 8.0045e-05, 5.5429e-02, 1.2788e-01, 3.8143e-07,
        1.4892e-06, 8.6914e-05, 1.8147e-04, 5.3603e-04, 6.3643e-04, 7.7748e-07,
        1.3245e-04, 2.1209e-03, 7.9427e-07, 1.7406e-04, 3.4672e-07, 8.1270e-05,
        2.0715e-02, 6.2921e-05, 1.5999e-05, 2.5794e-03, 4.2733e-08, 9.1594e-06,
        1.5908e-02, 1.8704e-05, 2.3311e-04, 8.3628e-06, 3.4470e-06, 8.1038e-03,
        9.1319e-04, 6.2826e-06, 1.2179e-03, 1.8739e-04, 9.4201e-01, 1.2773e-06,
        5.1104e-06, 3.4588e-07, 2.5803e-02, 2.5289e-03, 5.0191e-05, 1.8586e-04,
        3.1544e-05, 1.1483e-05, 7.8057e-07, 1.1968e-05, 1.1153e-06, 5.1968e-04,
        6.4925e-01, 5.4066e-07, 2.8354e-07, 2.3588e-07, 3.3020e-02, 2.6875e-04,
        4.3468e-03, 5.6795e-03, 1.8812e-05, 6.8205e-07, 1.4760e-07, 1.9405e-06,
        9.7793e-06, 2.4128e-04, 6.2857e-06, 9.9973e-01, 2.3406e-03, 1.5157e-03,
        1

In [ ]:
probs = F.softmax(logits, dim=1)                # [B, K]
if yb.dim() == 1:
    y_idx = yb.long()                           # [B]
    K = probs.size(1)
    y_onehot = F.one_hot(y_idx, num_classes=K)  # [B, K]

preds_idx = probs.argmax(dim=1)

# Overall accuracy (reuse existing key name)
acc = (preds_idx == y_idx).float().mean().item()

# Micro-F1 over class indices
f1_micro = f1_score(y_idx.cpu().numpy(), preds_idx.cpu().numpy(), average="micro")
f1_micro

0.8823529411764706

In [112]:
metrics

{'labelwise_acc': 0.8750104857075085,
 'f1_micro': 0.8750104856975086,
 'auroc': 0.9993252401637903,
 'auprc': 0.9431094482035531}

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import f1_score

num_classes = logits.shape[1]
y_true_oh = F.one_hot(yb.long(), num_classes=num_classes).cpu().numpy()

# NOTE: F.sigmoid is deprecated; use torch.sigmoid
y_pred_bin = (torch.sigmoid(logits).cpu().numpy() > 0.5).astype(int)

f1 = f1_score(y_true_oh, y_pred_bin, average="macro")

f1

/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0.04702947845804989

In [107]:
(F.sigmoid(logits)>0.5).cpu().numpy()

array([[False, False, False, ..., False, False, False],
       [False,  True, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [106]:
yb.cpu()

tensor([  5,   5,  32,  32,  32,  32, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167,
        167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167, 167])

In [103]:
f1_score([0,0,1],[0,1,1], average="macro")

0.6666666666666666

In [ ]:
yb

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')

In [59]:
metrics

{'labelwise_acc': 0.47849056689244396,
 'f1_micro': 0.4784905660377359,
 'auroc': 0.9873245828144305,
 'auprc': 0.9191004557418618}